# 🐄 Lumpy Skin Disease (LSD) Detection - Complete End-to-End Pipeline

**Project Goal**: Build a production-ready AI system to detect Lumpy Skin Disease in cattle using computer vision.

## 📋 Notebook Structure
1. **Environment Setup** - Install dependencies
2. **Dataset Collection** - Download and organize data
3. **Exploratory Data Analysis (EDA)** - Understand our data deeply
4. **Data Preprocessing & Cleaning** - Handle issues, standardize
5. **Data Augmentation Pipeline** - Robust training data
6. **Model Architecture** - Transfer learning with MobileNetV2/EfficientNet
7. **Training** - With proper validation strategy
8. **Evaluation** - Comprehensive metrics and visualization
9. **Model Explainability** - Grad-CAM for interpretability
10. **Optimization & Export** - Production-ready model

---

In [3]:
2+2

4

## 1️⃣ Environment Setup

In [4]:
# Install required packages
!pip install -q tensorflow>=2.13.0
!pip install -q pillow matplotlib seaborn pandas numpy scikit-learn
!pip install -q opencv-python-headless
!pip install -q kaggle  # For Kaggle dataset download
!pip install -q gdown   # For Google Drive downloads
!pip install -q albumentations  # Advanced augmentation
!pip install -q grad-cam  # For explainability

print("✅ All packages installed successfully!")

zsh:1: 2.13.0 not found
✅ All packages installed successfully!


In [5]:
pip install tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.7/200.7 MB 5.3 MB/s  0:00:38m0:00:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 5.3 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.9/676.9 kB 7.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 6.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 5.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 4.8 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15/15 [tensorflow]5 [tensorflow]]
Note: you may need to restart the kernel to use updated packages.


In [6]:
# Import all necessary libraries
import os
import sys
import random
import shutil
import warnings
from pathlib import Path
from collections import Counter

# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, 
    roc_curve, auc, precision_recall_curve
)
from sklearn.utils.class_weight import compute_class_weight

# Deep Learning - TensorFlow/Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.applications import (
    MobileNetV2, EfficientNetB0, EfficientNetV2S,
    ResNet50V2, DenseNet121
)

# Configuration
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

# GPU Configuration (if available)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✅ GPU Available: {len(gpus)} GPU(s) detected")
        print(f"   Device: {gpus}")
    except RuntimeError as e:
        print(f"⚠️ GPU configuration error: {e}")
else:
    print("⚠️ No GPU detected. Training will use CPU (slower).")

print(f"\n📦 TensorFlow version: {tf.__version__}")
print(f"🐍 Python version: {sys.version.split()[0]}")

: 

## 2️⃣ Dataset Collection & Organization

Based on research, we have multiple dataset sources:

### Available Datasets:
1. **Mendeley Data** - 1,024 images (700 healthy + 324 LSD) - Pre-processed, 256x256
2. **Kaggle** - Multiple datasets with varying sizes
3. **Roboflow** - 1,019 images (open source)

We'll start with the Mendeley dataset as it's well-documented and vetted.

In [1]:
# Configuration
PROJECT_DIR = Path('/home/claude/lsd_project')
DATA_DIR = PROJECT_DIR / 'data'
RAW_DATA_DIR = DATA_DIR / 'raw'
PROCESSED_DATA_DIR = DATA_DIR / 'processed'
MODELS_DIR = PROJECT_DIR / 'models'
RESULTS_DIR = PROJECT_DIR / 'results'

# Create directory structure
for directory in [PROJECT_DIR, DATA_DIR, RAW_DATA_DIR, PROCESSED_DATA_DIR, MODELS_DIR, RESULTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("✅ Directory structure created:")
print(f"   📁 Project root: {PROJECT_DIR}")
print(f"   📁 Data: {DATA_DIR}")
print(f"   📁 Models: {MODELS_DIR}")
print(f"   📁 Results: {RESULTS_DIR}")

NameError: name 'Path' is not defined

In [ ]:
# Dataset Download Instructions
print("""\n📥 DATASET DOWNLOAD OPTIONS:\n
Option 1: Mendeley Dataset (Recommended for starting)
---------------------------------------------------
1. Visit: https://data.mendeley.com/datasets/w36hpf86j2/1
2. Click 'Download All' (get ZIP file)
3. Extract to: {}
4. Structure should be:
   raw/
   ├── LUMPY SKIN/  (324 images)
   └── NORMAL SKIN/ (700 images)

Option 2: Kaggle Dataset (Alternative/Additional)
------------------------------------------------
1. Install Kaggle API: pip install kaggle
2. Setup API token from: https://www.kaggle.com/settings
3. Run: kaggle datasets download -d saurabhshahane/lumpy-skin-disease-dataset
4. Extract to raw/ directory

Option 3: Roboflow Dataset
--------------------------
Visit: https://universe.roboflow.com/qq-mgfrz/lumpy-skin-disease-detection

⚠️ IMPORTANT: After downloading, run the next cell to verify the dataset.
""".format(RAW_DATA_DIR))

In [ ]:
# For demonstration, we'll create a synthetic dataset structure
# In real implementation, you'll use actual downloaded data

# Check if dataset exists
def verify_dataset_structure(data_path):
    """
    Verify that dataset has proper structure.
    Expected structure:
    data_path/
    ├── class1/ (e.g., LUMPY SKIN or Lumpy)
    └── class2/ (e.g., NORMAL SKIN or Healthy)
    """
    data_path = Path(data_path)
    
    if not data_path.exists():
        print(f"❌ Dataset path not found: {data_path}")
        return False
    
    # Get subdirectories (classes)
    subdirs = [d for d in data_path.iterdir() if d.is_dir()]
    
    if len(subdirs) == 0:
        print(f"❌ No class folders found in {data_path}")
        return False
    
    print(f"\n✅ Dataset structure verified!")
    print(f"   📂 Total classes found: {len(subdirs)}")
    
    total_images = 0
    class_distribution = {}
    
    for subdir in subdirs:
        # Count images (common image extensions)
        image_files = list(subdir.glob('*.jpg')) + list(subdir.glob('*.jpeg')) + \
                      list(subdir.glob('*.png')) + list(subdir.glob('*.bmp'))
        
        count = len(image_files)
        class_distribution[subdir.name] = count
        total_images += count
        
        print(f"   📁 {subdir.name}: {count} images")
    
    print(f"\n   📊 Total images: {total_images}")
    
    return True, class_distribution, total_images

# Try to verify dataset
# Uncomment and adjust path after downloading dataset
# dataset_verified = verify_dataset_structure(RAW_DATA_DIR)

print("\n⚠️ Please download dataset first, then uncomment the verification line above.")

## 3️⃣ Exploratory Data Analysis (EDA)

Deep dive into our dataset to understand:
- Class distribution (imbalance?)
- Image properties (dimensions, file sizes, formats)
- Visual characteristics (brightness, contrast, color distribution)
- Quality issues (corrupted files, duplicates)
- Real-world challenges (lighting, angles, occlusions)

In [ ]:
def analyze_dataset(data_path):
    """
    Comprehensive dataset analysis
    """
    data_path = Path(data_path)
    
    # Collect all image information
    image_data = []
    
    for class_dir in data_path.iterdir():
        if not class_dir.is_dir():
            continue
            
        class_name = class_dir.name
        
        for img_path in class_dir.glob('*'):
            if img_path.suffix.lower() not in ['.jpg', '.jpeg', '.png', '.bmp']:
                continue
            
            try:
                # Load image
                img = Image.open(img_path)
                img_array = np.array(img)
                
                # Collect metadata
                image_data.append({
                    'path': str(img_path),
                    'class': class_name,
                    'filename': img_path.name,
                    'format': img.format,
                    'mode': img.mode,
                    'width': img.width,
                    'height': img.height,
                    'aspect_ratio': img.width / img.height,
                    'file_size_kb': img_path.stat().st_size / 1024,
                    'mean_brightness': np.mean(img_array),
                    'std_brightness': np.std(img_array),
                    'channels': img_array.shape[2] if len(img_array.shape) == 3 else 1
                })
                
            except Exception as e:
                print(f"⚠️ Error processing {img_path}: {e}")
    
    # Convert to DataFrame
    df = pd.DataFrame(image_data)
    
    return df

# This will be used after dataset is downloaded
# df_images = analyze_dataset(RAW_DATA_DIR)
# df_images.head()

In [ ]:
def plot_eda_visualizations(df):
    """
    Create comprehensive EDA visualizations
    """
    fig = plt.figure(figsize=(20, 12))
    
    # 1. Class Distribution
    ax1 = plt.subplot(2, 3, 1)
    class_counts = df['class'].value_counts()
    ax1.bar(class_counts.index, class_counts.values, color=['#FF6B6B', '#4ECDC4'])
    ax1.set_title('Class Distribution', fontsize=14, fontweight='bold')
    ax1.set_ylabel('Number of Images')
    for i, v in enumerate(class_counts.values):
        ax1.text(i, v + 10, str(v), ha='center', fontweight='bold')
    
    # Calculate imbalance ratio
    imbalance_ratio = class_counts.max() / class_counts.min()
    ax1.text(0.5, 0.95, f'Imbalance Ratio: {imbalance_ratio:.2f}:1', 
             transform=ax1.transAxes, ha='center', va='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    # 2. Image Dimensions Distribution
    ax2 = plt.subplot(2, 3, 2)
    ax2.scatter(df['width'], df['height'], alpha=0.5, c=df['class'].astype('category').cat.codes)
    ax2.set_xlabel('Width (pixels)')
    ax2.set_ylabel('Height (pixels)')
    ax2.set_title('Image Dimensions Distribution', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    
    # 3. Aspect Ratio Distribution
    ax3 = plt.subplot(2, 3, 3)
    for class_name in df['class'].unique():
        class_data = df[df['class'] == class_name]['aspect_ratio']
        ax3.hist(class_data, alpha=0.6, label=class_name, bins=20)
    ax3.set_xlabel('Aspect Ratio (Width/Height)')
    ax3.set_ylabel('Frequency')
    ax3.set_title('Aspect Ratio Distribution by Class', fontsize=14, fontweight='bold')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # 4. File Size Distribution
    ax4 = plt.subplot(2, 3, 4)
    ax4.boxplot([df[df['class'] == c]['file_size_kb'].values for c in df['class'].unique()],
                labels=df['class'].unique())
    ax4.set_ylabel('File Size (KB)')
    ax4.set_title('File Size Distribution by Class', fontsize=14, fontweight='bold')
    ax4.grid(True, alpha=0.3)
    
    # 5. Brightness Analysis
    ax5 = plt.subplot(2, 3, 5)
    for class_name in df['class'].unique():
        class_data = df[df['class'] == class_name]['mean_brightness']
        ax5.hist(class_data, alpha=0.6, label=class_name, bins=30)
    ax5.set_xlabel('Mean Brightness')
    ax5.set_ylabel('Frequency')
    ax5.set_title('Brightness Distribution by Class', fontsize=14, fontweight='bold')
    ax5.legend()
    ax5.grid(True, alpha=0.3)
    
    # 6. Summary Statistics Table
    ax6 = plt.subplot(2, 3, 6)
    ax6.axis('off')
    
    summary_stats = []
    for class_name in df['class'].unique():
        class_df = df[df['class'] == class_name]
        summary_stats.append([
            class_name,
            len(class_df),
            f"{class_df['width'].mean():.0f}x{class_df['height'].mean():.0f}",
            f"{class_df['file_size_kb'].mean():.1f}",
            f"{class_df['mean_brightness'].mean():.1f}"
        ])
    
    table = ax6.table(cellText=summary_stats,
                     colLabels=['Class', 'Count', 'Avg Size', 'Avg KB', 'Avg Brightness'],
                     cellLoc='center',
                     loc='center',
                     bbox=[0, 0, 1, 1])
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 2)
    ax6.set_title('Summary Statistics', fontsize=14, fontweight='bold', y=0.95)
    
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'eda_visualizations.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    return fig

# Will be used after dataset analysis
# plot_eda_visualizations(df_images)

In [ ]:
def display_sample_images(data_path, n_samples=5):
    """
    Display random sample images from each class
    """
    data_path = Path(data_path)
    classes = [d.name for d in data_path.iterdir() if d.is_dir()]
    
    fig, axes = plt.subplots(len(classes), n_samples, figsize=(20, 4 * len(classes)))
    
    for i, class_name in enumerate(classes):
        class_path = data_path / class_name
        image_files = list(class_path.glob('*.jpg')) + list(class_path.glob('*.png'))
        
        # Randomly sample images
        sample_images = random.sample(image_files, min(n_samples, len(image_files)))
        
        for j, img_path in enumerate(sample_images):
            img = Image.open(img_path)
            
            if len(classes) == 1:
                ax = axes[j]
            else:
                ax = axes[i, j]
            
            ax.imshow(img)
            ax.axis('off')
            
            if j == 0:
                ax.set_title(f'{class_name}\n{img.size[0]}x{img.size[1]}', 
                           fontsize=12, fontweight='bold')
            else:
                ax.set_title(f'{img.size[0]}x{img.size[1]}', fontsize=10)
    
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'sample_images.png', dpi=300, bbox_inches='tight')
    plt.show()

# Will be used after dataset download
# display_sample_images(RAW_DATA_DIR, n_samples=6)

## 4️⃣ Data Preprocessing & Cleaning

Critical steps:
1. Check for corrupted images
2. Remove duplicates
3. Standardize image formats
4. Handle class imbalance
5. Quality filtering (blur detection, too dark/bright)

In [ ]:
def check_image_quality(img_path, min_size=100, max_size=5000, 
                       blur_threshold=100, brightness_range=(20, 235)):
    """
    Check if image meets quality standards.
    Returns: (is_valid, reason)
    """
    try:
        img = Image.open(img_path)
        img_array = np.array(img)
        
        # Check 1: Image dimensions
        if img.width < min_size or img.height < min_size:
            return False, f"Too small: {img.width}x{img.height}"
        
        if img.width > max_size or img.height > max_size:
            return False, f"Too large: {img.width}x{img.height}"
        
        # Check 2: Corrupted or weird format
        if len(img_array.shape) not in [2, 3]:
            return False, "Invalid shape"
        
        # Check 3: Blur detection using Laplacian variance
        if len(img_array.shape) == 3:
            gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
        else:
            gray = img_array
        
        laplacian_var = cv2.Laplacian(gray, cv2.CV_64F).var()
        if laplacian_var < blur_threshold:
            return False, f"Too blurry: {laplacian_var:.2f}"
        
        # Check 4: Brightness
        mean_brightness = np.mean(img_array)
        if mean_brightness < brightness_range[0]:
            return False, f"Too dark: {mean_brightness:.1f}"
        if mean_brightness > brightness_range[1]:
            return False, f"Too bright: {mean_brightness:.1f}"
        
        return True, "OK"
        
    except Exception as e:
        return False, f"Error: {str(e)}"


def clean_dataset(input_dir, output_dir, quality_check=True):
    """
    Clean dataset by:
    1. Removing corrupted images
    2. Quality filtering
    3. Converting to standard format (RGB, JPEG)
    4. Removing duplicates
    """
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    stats = {
        'total': 0,
        'valid': 0,
        'corrupted': 0,
        'low_quality': 0,
        'duplicates': 0
    }
    
    # Track image hashes to detect duplicates
    image_hashes = set()
    
    for class_dir in input_dir.iterdir():
        if not class_dir.is_dir():
            continue
        
        class_name = class_dir.name
        output_class_dir = output_dir / class_name
        output_class_dir.mkdir(parents=True, exist_ok=True)
        
        print(f"\n🔍 Processing class: {class_name}")
        
        for img_path in class_dir.glob('*'):
            if img_path.suffix.lower() not in ['.jpg', '.jpeg', '.png', '.bmp']:
                continue
            
            stats['total'] += 1
            
            # Quality check
            if quality_check:
                is_valid, reason = check_image_quality(img_path)
                if not is_valid:
                    if "Error" in reason or "Invalid" in reason:
                        stats['corrupted'] += 1
                    else:
                        stats['low_quality'] += 1
                    print(f"   ❌ Rejected: {img_path.name} - {reason}")
                    continue
            
            try:
                # Load and standardize image
                img = Image.open(img_path)
                
                # Convert to RGB (handles grayscale, RGBA, etc.)
                if img.mode != 'RGB':
                    img = img.convert('RGB')
                
                # Check for duplicates using perceptual hash
                img_array = np.array(img.resize((8, 8)))  # Simple hash
                img_hash = hash(img_array.tobytes())
                
                if img_hash in image_hashes:
                    stats['duplicates'] += 1
                    print(f"   ⚠️ Duplicate: {img_path.name}")
                    continue
                
                image_hashes.add(img_hash)
                
                # Save standardized image
                output_path = output_class_dir / f"{img_path.stem}.jpg"
                img.save(output_path, 'JPEG', quality=95)
                
                stats['valid'] += 1
                
            except Exception as e:
                stats['corrupted'] += 1
                print(f"   ❌ Error processing {img_path.name}: {e}")
    
    # Print summary
    print("\n" + "="*60)
    print("📊 CLEANING SUMMARY")
    print("="*60)
    print(f"Total images processed: {stats['total']}")
    print(f"✅ Valid images: {stats['valid']} ({stats['valid']/stats['total']*100:.1f}%)")
    print(f"❌ Corrupted: {stats['corrupted']}")
    print(f"⚠️ Low quality: {stats['low_quality']}")
    print(f"🔄 Duplicates: {stats['duplicates']}")
    print("="*60)
    
    return stats

# Will be used after dataset download
# cleaning_stats = clean_dataset(RAW_DATA_DIR, PROCESSED_DATA_DIR, quality_check=True)

## 5️⃣ Data Splitting & Augmentation Pipeline

Create train/validation/test splits with proper stratification and augmentation.

In [ ]:
# Configuration for data pipeline
IMG_SIZE = 224  # Standard for MobileNetV2, can use 256/384 for EfficientNet
BATCH_SIZE = 32
TRAIN_SPLIT = 0.70
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15

print(f"""\n📊 DATA CONFIGURATION:
   Image Size: {IMG_SIZE}x{IMG_SIZE}
   Batch Size: {BATCH_SIZE}
   Split Ratio: Train={TRAIN_SPLIT:.0%} | Val={VAL_SPLIT:.0%} | Test={TEST_SPLIT:.0%}
""")

In [ ]:
def create_train_val_test_split(data_dir, output_dir, train_size=0.7, val_size=0.15, test_size=0.15):
    """
    Split dataset into train/val/test with stratification
    """
    data_dir = Path(data_dir)
    output_dir = Path(output_dir)
    
    # Create split directories
    for split in ['train', 'val', 'test']:
        (output_dir / split).mkdir(parents=True, exist_ok=True)
    
    split_stats = {}
    
    for class_dir in data_dir.iterdir():
        if not class_dir.is_dir():
            continue
        
        class_name = class_dir.name
        print(f"\n📂 Splitting class: {class_name}")
        
        # Create class directories in each split
        for split in ['train', 'val', 'test']:
            (output_dir / split / class_name).mkdir(parents=True, exist_ok=True)
        
        # Get all images
        image_files = list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.png'))
        
        # Shuffle
        random.shuffle(image_files)
        
        # Calculate split indices
        n_total = len(image_files)
        n_train = int(n_total * train_size)
        n_val = int(n_total * val_size)
        
        # Split files
        train_files = image_files[:n_train]
        val_files = image_files[n_train:n_train + n_val]
        test_files = image_files[n_train + n_val:]
        
        split_stats[class_name] = {
            'total': n_total,
            'train': len(train_files),
            'val': len(val_files),
            'test': len(test_files)
        }
        
        # Copy files to respective directories
        for img_file in train_files:
            shutil.copy2(img_file, output_dir / 'train' / class_name / img_file.name)
        
        for img_file in val_files:
            shutil.copy2(img_file, output_dir / 'val' / class_name / img_file.name)
        
        for img_file in test_files:
            shutil.copy2(img_file, output_dir / 'test' / class_name / img_file.name)
        
        print(f"   Train: {len(train_files)} | Val: {len(val_files)} | Test: {len(test_files)}")
    
    # Print summary
    print("\n" + "="*60)
    print("📊 SPLIT SUMMARY")
    print("="*60)
    
    df_stats = pd.DataFrame(split_stats).T
    print(df_stats)
    print("\n" + "="*60)
    
    return split_stats

# Will be used after cleaning
# SPLIT_DATA_DIR = DATA_DIR / 'split'
# split_stats = create_train_val_test_split(PROCESSED_DATA_DIR, SPLIT_DATA_DIR, 
#                                           train_size=TRAIN_SPLIT, 
#                                           val_size=VAL_SPLIT, 
#                                           test_size=TEST_SPLIT)

In [ ]:
# Data Augmentation Configuration

# Training augmentation - aggressive
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=False,  # Cows are usually upright
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

# Validation and Test - only rescaling (no augmentation)
val_test_datagen = ImageDataGenerator(rescale=1./255)

print("✅ Data augmentation pipelines configured!")
print("   Training: Rotation, shift, zoom, flip, brightness variation")
print("   Val/Test: Only normalization (no augmentation)")

In [ ]:
def create_data_generators(split_dir, img_size=224, batch_size=32):
    """
    Create data generators for train/val/test
    """
    split_dir = Path(split_dir)
    
    # Training generator with augmentation
    train_generator = train_datagen.flow_from_directory(
        split_dir / 'train',
        target_size=(img_size, img_size),
        batch_size=batch_size,
        class_mode='categorical',
        shuffle=True,
        seed=SEED
    )
    
    # Validation generator (no augmentation)
    val_generator = val_test_datagen.flow_from_directory(
        split_dir / 'val',
        target_size=(img_size, img_size),
        batch_size=batch_size,
        class_mode='categorical',
        shuffle=False
    )
    
    # Test generator (no augmentation)
    test_generator = val_test_datagen.flow_from_directory(
        split_dir / 'test',
        target_size=(img_size, img_size),
        batch_size=batch_size,
        class_mode='categorical',
        shuffle=False
    )
    
    print("\n✅ Data generators created:")
    print(f"   Train samples: {train_generator.samples}")
    print(f"   Val samples: {val_generator.samples}")
    print(f"   Test samples: {test_generator.samples}")
    print(f"   Classes: {train_generator.class_indices}")
    
    return train_generator, val_generator, test_generator

# Will be used after splitting
# train_gen, val_gen, test_gen = create_data_generators(SPLIT_DATA_DIR, IMG_SIZE, BATCH_SIZE)

In [ ]:
def visualize_augmentations(data_dir, class_name, n_augmentations=5):
    """
    Visualize what augmentations do to images
    """
    data_dir = Path(data_dir)
    class_dir = data_dir / 'train' / class_name
    
    # Get random image
    img_files = list(class_dir.glob('*.jpg'))
    sample_img = random.choice(img_files)
    
    # Load image
    img = load_img(sample_img, target_size=(IMG_SIZE, IMG_SIZE))
    img_array = img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    
    # Create figure
    fig, axes = plt.subplots(1, n_augmentations + 1, figsize=(20, 4))
    
    # Original image
    axes[0].imshow(img)
    axes[0].set_title('Original', fontsize=12, fontweight='bold')
    axes[0].axis('off')
    
    # Generate augmented versions
    i = 1
    for batch in train_datagen.flow(img_array, batch_size=1):
        axes[i].imshow(batch[0])
        axes[i].set_title(f'Augmented {i}', fontsize=12)
        axes[i].axis('off')
        i += 1
        if i > n_augmentations:
            break
    
    plt.suptitle(f'Augmentation Examples - {class_name}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f'augmentation_examples_{class_name}.png', dpi=300, bbox_inches='tight')
    plt.show()

# Will be used after data preparation
# visualize_augmentations(SPLIT_DATA_DIR, 'LUMPY SKIN', n_augmentations=6)

---

## 🎯 CHECKPOINT: Data Preparation Complete!

Before moving to model training, ensure:
- ✅ Dataset downloaded and verified
- ✅ EDA completed (understand class distribution, image properties)
- ✅ Data cleaned (corrupted/low-quality images removed)
- ✅ Train/Val/Test split created
- ✅ Data generators configured with augmentation

**Next Steps:** Model architecture selection and training!

---

## 6️⃣ Model Architecture

We'll implement multiple architectures and compare:
1. **MobileNetV2** - Fast, lightweight (baseline)
2. **EfficientNetV2-S** - Better accuracy, still mobile-friendly
3. **ResNet50V2** - Solid performance

All using transfer learning from ImageNet weights.

In [ ]:
def create_model(architecture='mobilenetv2', num_classes=2, img_size=224, trainable_layers=20):
    """
    Create transfer learning model.
    
    Args:
        architecture: 'mobilenetv2', 'efficientnetv2s', 'resnet50v2', 'densenet121'
        num_classes: Number of output classes
        img_size: Input image size
        trainable_layers: Number of layers to fine-tune (from the end)
    """
    input_shape = (img_size, img_size, 3)
    
    # Select base model
    if architecture == 'mobilenetv2':
        base_model = MobileNetV2(
            input_shape=input_shape,
            include_top=False,
            weights='imagenet'
        )
    elif architecture == 'efficientnetv2s':
        base_model = EfficientNetV2S(
            input_shape=input_shape,
            include_top=False,
            weights='imagenet'
        )
    elif architecture == 'efficientnetb0':
        base_model = EfficientNetB0(
            input_shape=input_shape,
            include_top=False,
            weights='imagenet'
        )
    elif architecture == 'resnet50v2':
        base_model = ResNet50V2(
            input_shape=input_shape,
            include_top=False,
            weights='imagenet'
        )
    elif architecture == 'densenet121':
        base_model = DenseNet121(
            input_shape=input_shape,
            include_top=False,
            weights='imagenet'
        )
    else:
        raise ValueError(f"Unknown architecture: {architecture}")
    
    # Freeze base model initially
    base_model.trainable = False
    
    # Build model
    inputs = keras.Input(shape=input_shape)
    
    # Preprocessing (if needed by specific architecture)
    x = inputs
    
    # Base model
    x = base_model(x, training=False)
    
    # Classification head
    x = layers.GlobalAveragePooling2D(name='global_avg_pool')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3, name='dropout_1')(x)
    x = layers.Dense(256, activation='relu', name='dense_1')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3, name='dropout_2')(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='predictions')(x)
    
    # Create model
    model = keras.Model(inputs, outputs, name=f'lsd_detector_{architecture}')
    
    print(f"\n✅ Model created: {architecture.upper()}")
    print(f"   Total parameters: {model.count_params():,}")
    print(f"   Base model layers: {len(base_model.layers)}")
    
    return model, base_model

# Example usage
# model, base_model = create_model('mobilenetv2', num_classes=2, img_size=IMG_SIZE)

In [ ]:
def compile_model(model, learning_rate=1e-4):
    """
    Compile model with optimizer and loss function
    """
    # Optimizer
    optimizer = optimizers.Adam(learning_rate=learning_rate)
    
    # Loss function
    # Using categorical crossentropy (can switch to focal loss for imbalance)
    loss = keras.losses.CategoricalCrossentropy(label_smoothing=0.1)
    
    # Metrics
    metrics = [
        'accuracy',
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall'),
        keras.metrics.AUC(name='auc')
    ]
    
    model.compile(
        optimizer=optimizer,
        loss=loss,
        metrics=metrics
    )
    
    print("\n✅ Model compiled!")
    print(f"   Optimizer: Adam (lr={learning_rate})")
    print(f"   Loss: Categorical Crossentropy (label_smoothing=0.1)")
    print(f"   Metrics: {[m if isinstance(m, str) else m.name for m in metrics]}")
    
    return model

# Example
# model = compile_model(model, learning_rate=1e-4)

## 7️⃣ Training Strategy

Two-phase training:
1. **Phase 1**: Train only classification head (base frozen)
2. **Phase 2**: Fine-tune top layers of base model

In [ ]:
# Training configuration
EPOCHS_PHASE1 = 15  # Train classifier head
EPOCHS_PHASE2 = 25  # Fine-tune with base model
LEARNING_RATE_PHASE1 = 1e-3
LEARNING_RATE_PHASE2 = 1e-5

# Callbacks
def get_callbacks(model_name='model', patience=7):
    """
    Define training callbacks
    """
    callbacks_list = [
        # Early stopping
        callbacks.EarlyStopping(
            monitor='val_loss',
            patience=patience,
            restore_best_weights=True,
            verbose=1
        ),
        
        # Model checkpoint
        callbacks.ModelCheckpoint(
            filepath=str(MODELS_DIR / f'{model_name}_best.keras'),
            monitor='val_accuracy',
            save_best_only=True,
            verbose=1
        ),
        
        # Reduce learning rate on plateau
        callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=3,
            min_lr=1e-7,
            verbose=1
        ),
        
        # TensorBoard (optional)
        # callbacks.TensorBoard(
        #     log_dir=str(RESULTS_DIR / 'logs'),
        #     histogram_freq=1
        # )
    ]
    
    return callbacks_list

print("✅ Callbacks configured: EarlyStopping, ModelCheckpoint, ReduceLROnPlateau")

In [ ]:
def train_model_phase1(model, train_gen, val_gen, epochs=15, learning_rate=1e-3):
    """
    Phase 1: Train only the classification head
    """
    print("\n" + "="*70)
    print("🚀 PHASE 1: Training Classification Head (Base Model Frozen)")
    print("="*70)
    
    # Compile model
    model = compile_model(model, learning_rate=learning_rate)
    
    # Get callbacks
    callbacks_list = get_callbacks(model_name='phase1', patience=5)
    
    # Train
    history_phase1 = model.fit(
        train_gen,
        epochs=epochs,
        validation_data=val_gen,
        callbacks=callbacks_list,
        verbose=1
    )
    
    print("\n✅ Phase 1 training complete!")
    
    return model, history_phase1


def train_model_phase2(model, base_model, train_gen, val_gen, 
                       epochs=25, learning_rate=1e-5, unfreeze_layers=30):
    """
    Phase 2: Fine-tune top layers of base model
    """
    print("\n" + "="*70)
    print("🔥 PHASE 2: Fine-tuning Top Layers of Base Model")
    print("="*70)
    
    # Unfreeze top layers
    base_model.trainable = True
    
    # Freeze all except top N layers
    for layer in base_model.layers[:-unfreeze_layers]:
        layer.trainable = False
    
    trainable_count = sum([1 for layer in base_model.layers if layer.trainable])
    print(f"   Trainable layers in base model: {trainable_count}/{len(base_model.layers)}")
    
    # Recompile with lower learning rate
    model = compile_model(model, learning_rate=learning_rate)
    
    # Get callbacks
    callbacks_list = get_callbacks(model_name='phase2_final', patience=7)
    
    # Train
    history_phase2 = model.fit(
        train_gen,
        epochs=epochs,
        validation_data=val_gen,
        callbacks=callbacks_list,
        verbose=1
    )
    
    print("\n✅ Phase 2 training complete!")
    print(f"   Final model saved to: {MODELS_DIR / 'phase2_final_best.keras'}")
    
    return model, history_phase2

# Training will be executed after data preparation
# model, history1 = train_model_phase1(model, train_gen, val_gen, 
#                                      epochs=EPOCHS_PHASE1, 
#                                      learning_rate=LEARNING_RATE_PHASE1)
# 
# model, history2 = train_model_phase2(model, base_model, train_gen, val_gen,
#                                      epochs=EPOCHS_PHASE2,
#                                      learning_rate=LEARNING_RATE_PHASE2)

## 8️⃣ Model Evaluation & Visualization

Comprehensive evaluation with multiple metrics

In [ ]:
def plot_training_history(history1, history2=None):
    """
    Plot training and validation metrics over epochs
    """
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    metrics = ['loss', 'accuracy', 'precision', 'recall']
    titles = ['Loss', 'Accuracy', 'Precision', 'Recall']
    
    for idx, (metric, title) in enumerate(zip(metrics, titles)):
        ax = axes[idx // 2, idx % 2]
        
        # Phase 1
        epochs1 = range(1, len(history1.history[metric]) + 1)
        ax.plot(epochs1, history1.history[metric], 'b-', label='Phase 1 Train', linewidth=2)
        ax.plot(epochs1, history1.history[f'val_{metric}'], 'b--', label='Phase 1 Val', linewidth=2)
        
        # Phase 2 (if provided)
        if history2:
            offset = len(epochs1)
            epochs2 = range(offset + 1, offset + len(history2.history[metric]) + 1)
            ax.plot(epochs2, history2.history[metric], 'r-', label='Phase 2 Train', linewidth=2)
            ax.plot(epochs2, history2.history[f'val_{metric}'], 'r--', label='Phase 2 Val', linewidth=2)
            
            # Add vertical line to separate phases
            ax.axvline(x=offset, color='gray', linestyle=':', linewidth=2, label='Phase Transition')
        
        ax.set_title(f'{title} Over Epochs', fontsize=14, fontweight='bold')
        ax.set_xlabel('Epoch')
        ax.set_ylabel(title)
        ax.legend(loc='best')
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'training_history.png', dpi=300, bbox_inches='tight')
    plt.show()

# Usage after training
# plot_training_history(history1, history2)

In [ ]:
def evaluate_model(model, test_gen, class_names):
    """
    Comprehensive model evaluation on test set
    """
    print("\n" + "="*70)
    print("📊 MODEL EVALUATION ON TEST SET")
    print("="*70)
    
    # Get predictions
    test_gen.reset()
    predictions = model.predict(test_gen, verbose=1)
    predicted_classes = np.argmax(predictions, axis=1)
    true_classes = test_gen.classes
    
    # Classification report
    print("\n📋 Classification Report:")
    print("="*70)
    report = classification_report(true_classes, predicted_classes, 
                                   target_names=class_names, digits=4)
    print(report)
    
    # Confusion matrix
    cm = confusion_matrix(true_classes, predicted_classes)
    
    # Calculate additional metrics
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
    
    accuracy = accuracy_score(true_classes, predicted_classes)
    precision = precision_score(true_classes, predicted_classes, average='weighted')
    recall = recall_score(true_classes, predicted_classes, average='weighted')
    f1 = f1_score(true_classes, predicted_classes, average='weighted')
    
    print("\n📈 Overall Metrics:")
    print("="*70)
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    
    # CRITICAL METRICS FOR LSD DETECTION
    # Assuming class 0 = Healthy, class 1 = Lumpy
    # For medical diagnosis, we care most about:
    # - Sensitivity (True Positive Rate) - catch all LSD cases
    # - Specificity (True Negative Rate) - don't alarm farmers unnecessarily
    
    tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)
    
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0  # Recall for positive class
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    print("\n🎯 CRITICAL METRICS (for LSD detection):")
    print("="*70)
    print(f"Sensitivity (LSD Detection Rate): {sensitivity:.4f} - Target: >0.95")
    print(f"Specificity (Healthy Accuracy):   {specificity:.4f} - Target: >0.90")
    print(f"False Negatives: {fn} (CRITICAL - missed LSD cases)")
    print(f"False Positives: {fp} (alerts for healthy cattle)")
    
    return {
        'predictions': predictions,
        'predicted_classes': predicted_classes,
        'true_classes': true_classes,
        'confusion_matrix': cm,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'sensitivity': sensitivity,
        'specificity': specificity
    }

# Usage after training
# class_names = list(train_gen.class_indices.keys())
# eval_results = evaluate_model(model, test_gen, class_names)

In [ ]:
def plot_confusion_matrix(cm, class_names):
    """
    Plot detailed confusion matrix
    """
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Normalize confusion matrix
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    # Create heatmap
    sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                cbar_kws={'label': 'Percentage'},
                square=True, linewidths=1, ax=ax)
    
    # Add counts
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            text = ax.text(j + 0.5, i + 0.7, f'({cm[i, j]})',
                          ha="center", va="center", color="red", fontsize=10)
    
    ax.set_ylabel('True Label', fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
    ax.set_title('Confusion Matrix (Normalized with Counts)', fontsize=14, fontweight='bold', pad=20)
    
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'confusion_matrix.png', dpi=300, bbox_inches='tight')
    plt.show()

# Usage
# plot_confusion_matrix(eval_results['confusion_matrix'], class_names)

In [ ]:
def plot_roc_curve(eval_results, class_names):
    """
    Plot ROC curve for each class
    """
    from sklearn.preprocessing import label_binarize
    
    # Binarize labels for multi-class ROC
    y_test_bin = label_binarize(eval_results['true_classes'], 
                                 classes=range(len(class_names)))
    y_pred_proba = eval_results['predictions']
    
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Plot ROC curve for each class
    for i, class_name in enumerate(class_names):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_pred_proba[:, i])
        roc_auc = auc(fpr, tpr)
        
        ax.plot(fpr, tpr, linewidth=2, 
               label=f'{class_name} (AUC = {roc_auc:.3f})')
    
    # Plot diagonal (random classifier)
    ax.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Random Classifier')
    
    ax.set_xlabel('False Positive Rate', fontsize=12, fontweight='bold')
    ax.set_ylabel('True Positive Rate', fontsize=12, fontweight='bold')
    ax.set_title('ROC Curves', fontsize=14, fontweight='bold')
    ax.legend(loc='lower right', fontsize=11)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'roc_curves.png', dpi=300, bbox_inches='tight')
    plt.show()

# Usage
# plot_roc_curve(eval_results, class_names)

## 🎯 Complete Training & Evaluation Template

Once you have your dataset ready, uncomment and run this section to execute the full pipeline.

In [ ]:
# COMPLETE PIPELINE - UNCOMMENT AFTER DATASET PREPARATION

# # 1. Prepare data
# SPLIT_DATA_DIR = DATA_DIR / 'split'
# split_stats = create_train_val_test_split(PROCESSED_DATA_DIR, SPLIT_DATA_DIR)
# 
# # 2. Create data generators
# train_gen, val_gen, test_gen = create_data_generators(SPLIT_DATA_DIR, IMG_SIZE, BATCH_SIZE)
# class_names = list(train_gen.class_indices.keys())
# num_classes = len(class_names)
# 
# # 3. Create model
# model, base_model = create_model('mobilenetv2', num_classes=num_classes, img_size=IMG_SIZE)
# 
# # 4. Phase 1 training (classifier head)
# model, history1 = train_model_phase1(
#     model, train_gen, val_gen,
#     epochs=EPOCHS_PHASE1,
#     learning_rate=LEARNING_RATE_PHASE1
# )
# 
# # 5. Phase 2 training (fine-tuning)
# model, history2 = train_model_phase2(
#     model, base_model, train_gen, val_gen,
#     epochs=EPOCHS_PHASE2,
#     learning_rate=LEARNING_RATE_PHASE2,
#     unfreeze_layers=30
# )
# 
# # 6. Plot training history
# plot_training_history(history1, history2)
# 
# # 7. Evaluate on test set
# eval_results = evaluate_model(model, test_gen, class_names)
# 
# # 8. Visualization
# plot_confusion_matrix(eval_results['confusion_matrix'], class_names)
# plot_roc_curve(eval_results, class_names)
# 
# print("\n✅ Training and evaluation complete!")
# print(f"   Final model saved at: {MODELS_DIR / 'phase2_final_best.keras'}")

## 9️⃣ Model Explainability (Grad-CAM)

Visualize which parts of the image the model focuses on - critical for trust and debugging.

In [ ]:
# Grad-CAM implementation will be added in next iteration
# This helps show farmers WHY the model detected LSD

print("\n🔬 Grad-CAM implementation placeholder")
print("   This will highlight disease areas in the image")
print("   Coming in the next iteration!")

## 🔟 Model Export & Optimization

Prepare model for deployment (mobile, edge devices)

In [ ]:
def export_model_for_deployment(model, model_name='lsd_detector'):
    """
    Export model in multiple formats for deployment
    """
    print("\n📦 Exporting model for deployment...")
    
    # 1. SavedModel format (for TensorFlow Serving)
    saved_model_path = MODELS_DIR / f'{model_name}_saved_model'
    model.save(saved_model_path)
    print(f"   ✅ SavedModel: {saved_model_path}")
    
    # 2. TFLite (for mobile/edge)
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_model = converter.convert()
    
    tflite_path = MODELS_DIR / f'{model_name}.tflite'
    with open(tflite_path, 'wb') as f:
        f.write(tflite_model)
    
    print(f"   ✅ TFLite: {tflite_path}")
    print(f"      Size: {tflite_path.stat().st_size / 1024 / 1024:.2f} MB")
    
    # 3. Quantized TFLite (even smaller)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.target_spec.supported_types = [tf.float16]
    tflite_quant_model = converter.convert()
    
    tflite_quant_path = MODELS_DIR / f'{model_name}_quantized.tflite'
    with open(tflite_quant_path, 'wb') as f:
        f.write(tflite_quant_model)
    
    print(f"   ✅ Quantized TFLite: {tflite_quant_path}")
    print(f"      Size: {tflite_quant_path.stat().st_size / 1024 / 1024:.2f} MB")
    
    print("\n✅ Model export complete!")
    
    return {
        'saved_model': saved_model_path,
        'tflite': tflite_path,
        'tflite_quantized': tflite_quant_path
    }

# Usage after training
# export_paths = export_model_for_deployment(model, 'lsd_detector_v1')

---

## 📝 Summary & Next Steps

### What we've built:
1. ✅ Complete data pipeline (collection → cleaning → augmentation)
2. ✅ Transfer learning architecture (MobileNetV2 baseline)
3. ✅ Two-phase training strategy
4. ✅ Comprehensive evaluation metrics
5. ✅ Model export for deployment

### To execute this notebook:
1. Download dataset (Mendeley/Kaggle)
2. Uncomment verification and analysis cells
3. Run data cleaning and splitting
4. Execute training pipeline
5. Evaluate and export model

### Next iterations will add:
- 🔬 Grad-CAM explainability
- 📱 Mobile app integration code
- 🤖 Chatbot for post-diagnosis guidance
- 🌐 API for cloud deployment
- 📊 Real-world performance monitoring

---